# TSA Chapter 12: Wavelet Coherence Between Two Economic Series

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/TSA/blob/main/TSA_ch12/TSA_ch12_wavelet_coherence/TSA_ch12_wavelet_coherence.ipynb)

In [ ]:
!pip install matplotlib numpy scipy statsmodels PyWavelets -q

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pywt
from scipy.ndimage import gaussian_filter

In [ ]:
# Style configuration
COLORS = {
    'blue':   '#1A3A6E',
    'red':    '#DC3545',
    'green':  '#2E7D32',
    'orange': '#E67E22',
    'gray':   '#666666',
    'purple': '#8E44AD',
}

plt.rcParams.update({
    'axes.facecolor':     'none',
    'figure.facecolor':   'none',
    'savefig.transparent': True,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          False,
    'font.size':          9,
    'axes.labelsize':     9,
    'axes.titlesize':     10,
    'legend.fontsize':    8,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
})

In [ ]:
# Chart: ch12_coherence_example
# Wavelet coherence map: time-varying co-movement between two series
rng = np.random.default_rng(42)
N = 512
t = np.linspace(0, 50, N)
f1, f2 = 0.2, 0.5
X = np.where(t < 25, np.sin(2*np.pi*f1*t), np.sin(2*np.pi*f2*t)) + 0.3*rng.normal(size=N)
Y = np.where(t < 25, np.sin(2*np.pi*f1*(t-1)), np.sin(2*np.pi*f2*(t-2))) + 0.3*rng.normal(size=N)

wavelet = 'cmor1.5-1.0'
scales = np.geomspace(4, 256, 60)
dt = t[1] - t[0]
cX, _ = pywt.cwt(X, scales, wavelet, sampling_period=dt)
cY, freqs = pywt.cwt(Y, scales, wavelet, sampling_period=dt)

def smooth2d(W, sigma=2): return gaussian_filter(np.abs(W)**2, sigma=sigma)
cross_r = gaussian_filter(np.real(cX * np.conj(cY)), sigma=2)
cross_i = gaussian_filter(np.imag(cX * np.conj(cY)), sigma=2)
cross = cross_r + 1j * cross_i
WCO = np.clip(np.abs(cross)**2 / (smooth2d(cX) * smooth2d(cY) + 1e-12), 0, 1)

fig, axes = plt.subplots(3, 1, figsize=(11, 9), gridspec_kw={'height_ratios': [1, 1, 2.5]})
axes[0].plot(t, X, color=COLORS['blue'], lw=1)
axes[0].set_title('Series X'); axes[0].set_ylabel('X')
axes[1].plot(t, Y, color=COLORS['red'], lw=1)
axes[1].set_title('Series Y'); axes[1].set_ylabel('Y')
im = axes[2].contourf(t, freqs, WCO, levels=20, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(im, ax=axes[2], label='Wavelet Coherence')
axes[2].set_title('Wavelet Coherence |W_XY|² / (S_XX · S_YY)')
axes[2].set_ylabel('Frequency (Hz)'); axes[2].set_xlabel('Time')
axes[2].set_yscale('log')

for ax in axes:
    ax.set_facecolor('none')
    ax.spines[['top','right']].set_visible(False)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.savefig('ch12_coherence_example.pdf', bbox_inches='tight', dpi=150)
plt.savefig('ch12_coherence_example.png', bbox_inches='tight', dpi=150)
plt.show()